In [ ]:
import numpy as np
import pandas as pd
import os

## Download data from Retrosheet

In [7]:
import requests

os.makedirs('../data/zips', exist_ok=True)

url = 'https://www.retrosheet.org/downloads/plays/2025plays.zip'
file_path = '../data/zips/2025plays.zip' # NOTE: Path for non-notebooks

try:
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, 'wb') as file:
        file.write(response.content)

except Exception as e:
    print(e)

### Extract the zip

In [10]:
import zipfile

with zipfile.ZipFile('../data/zips/2025plays.zip', 'r') as zip_ref:
    zip_ref.extractall('../data/plays')

## Load the DF

In [97]:
df = pd.read_csv('../data/plays/2025plays.csv')
df = df[df['gametype'] == 'regular']

/var/folders/fz/tl6wl4zn14d896djssv9plyr0000gn/T/ipykernel_22936/822759290.py:1: DtypeWarning: Columns (0: umplf, 1: umprf) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/plays/2025plays.csv')


## Pre Format Data

In [211]:
def get_pa_result(row):
    if row['single'] == 1:
        return '1b'
    
    if row['double'] == 1:
        return '2b'
    
    if row['triple'] == 1:
        return '3b'
    
    if row['hr'] == 1:
        return 'hr'
    
    if row['sh'] == 1:
        return None
    
    # Trying this as an out instead
    #
    # As its own thing, it's an out plus a run went in. If a RP
    # comes in and gets an out, but an unearned-run scores, that's not really on him
    #
    # if row['sf'] == 1:
    #     return 'sf'
    
    if row['hbp'] == 1:
        return 'hbp'
    
    if row['walk'] == 1:
        return 'walk'
    
    if row['k'] == 1:
        return 'k'
    
    if row['xi'] == 1:
        return None
    
    if row['roe'] == 1:
        return None
    
    # Trying this as an out instead
    #
    # As its own thing, it's an out but a runner was on base. A pitcher would have
    # had an out against the batter, but a fielder "interfered"
    #
    # if row['fc'] == 1:
    #     return 'fc'
    
    return 'other_out'

df['pa_result'] = df.apply(get_pa_result, axis=1)
df = df.loc[df['pa_result'].notna()]
df.head()

,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp,pa_result
0,CHN202503180,43/G34,1,0,0,TOK01,LAN,CHN,0,0,...,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full,other_out
1,CHN202503180,3/P34,1,0,0,TOK01,LAN,CHN,0,0,...,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full,other_out
2,CHN202503180,K,1,0,0,TOK01,LAN,CHN,0,0,...,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full,k
3,CHN202503180,W,1,1,1,TOK01,CHN,LAN,0,0,...,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full,walk
4,CHN202503180,6/L6MD,1,1,1,TOK01,CHN,LAN,0,0,...,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full,other_out


## Create RE Matrix

In [213]:
def get_state(row, pre_post):
    bases = ''
    
    bases += '-' if pd.isna(row[f'br1_{pre_post}']) else '1'
    bases += '-' if pd.isna(row[f'br2_{pre_post}']) else '2'
    bases += '-' if pd.isna(row[f'br3_{pre_post}']) else '3'
    
    if bases == '---':
        return 0 + (8 * row[f'outs_{pre_post}'])
    elif bases == '1--':
        return 1 + (8 * row[f'outs_{pre_post}'])
    elif bases == '-2-':
        return 2 + (8 * row[f'outs_{pre_post}'])
    elif bases == '--3':
        return 3 + (8 * row[f'outs_{pre_post}'])
    elif bases == '12-':
        return 4 + (8 * row[f'outs_{pre_post}'])
    elif bases == '1-3':
        return 5 + (8 * row[f'outs_{pre_post}'])
    elif bases == '-23':
        return 6 + (8 * row[f'outs_{pre_post}'])
    elif bases == '123':
        return 7 + (8 * row[f'outs_{pre_post}'])


    return 'XXX'


df_re = df[[
    'gid',
    'inning',
    'top_bot',
    'batter',
    'pitcher',
    'pa_result',
    'outs_pre',
    'outs_post',
    'br1_pre',
    'br2_pre',
    'br3_pre',
    'br1_post',
    'br2_post',
    'br3_post',
    'runs'
]]

# NOTE: Consider that these should generally be sorted, but be mindful that they may not be
df_re['inning_id'] = df_re.apply(lambda x: f'{x['gid']}_{x['inning']}_{x['top_bot']}', axis=1)
df_re = df_re.drop(columns=['gid', 'inning', 'top_bot'])

df_re['re_state_pre']  = df_re.apply(get_state, pre_post='pre', axis=1)
df_re = df_re[df_re['re_state_pre'] != 24]
df_re['re_state_post'] = df_re.apply(get_state, pre_post='post', axis=1)

df_re = df_re.drop(columns=[f'br1_pre', 'br2_pre', 'br3_pre', 'br1_post', 'br2_post', 'br3_post'])

df_re['runs_remaining_in_inning'] = (
    df_re.groupby('inning_id')['runs']
      .transform(lambda x: x[::-1].cumsum()[::-1])
)

re_matrix = (
    df_re.groupby('re_state_pre')['runs_remaining_in_inning']
      .mean()
      .round(3)
)

# re_matrix = pd.DataFrame(
#     re_matrix.values.reshape(3, 8),
#     index=pd.Index([0, 1, 2], name='outs'),
#     columns=pd.Index(['___','1__','_2_','__3','12_','1_3','_23','123'], name='base_state')
# )

re_matrix

re_state_pre
0     0.487
1     0.879
2     1.113
3     1.308
4     1.564
5     1.801
6     1.933
7     2.433
8     0.262
9     0.513
10    0.642
11    0.879
12    0.944
13    1.207
14    1.382
15    1.573
16    0.101
17    0.223
18    0.313
19    0.331
20    0.454
21    0.468
22    0.567
23    0.743
Name: runs_remaining_in_inning, dtype: float64

## Get wOBA Values

In [214]:

df_woba = df_re.drop(columns=['inning_id'])
df_woba['re_pre']  = df_woba.apply(lambda row: re_matrix[row['re_state_pre']], axis=1)
df_woba['re_post'] = df_woba.apply(lambda row: 0 + row['runs'] if row['re_state_post'] == 24 else re_matrix[row['re_state_post']] + row['runs'], axis=1)
df_woba['re_diff'] = df_woba['re_post'] - df_woba['re_pre']
woba_c = df_woba.groupby('pa_result')['re_diff'].mean()
woba_c

pa_result
1b           0.469319
2b           0.747943
3b           0.993170
hbp          0.352305
hr           1.387305
k           -0.268650
other_out   -0.236677
walk         0.326421
Name: re_diff, dtype: float64

## ELO v0.1

### Normalize wOBA Coefficients

In [216]:
wobas = woba_c.to_dict()

w_min = min(wobas.values())
w_max = max(wobas.values())
OUTCOMES = {
        k: (v - w_min) / (w_max - w_min)
        for k, v in wobas.items()
    }

# OUTCOMES

LG_AVG_PA_MIX = df_woba['pa_result'].value_counts(normalize=True).to_dict()

league_average = sum(
    OUTCOMES[k] * v for k, v in LG_AVG_PA_MIX.items()
)
# print(LG_AVG_PA_MIX)
# print('='*10)
# print(league_average)

In [ ]:
batters = {}
pitchers = {}

i = 0

for index, row in df_woba.iterrows():
    batter_id = row['batter']
    pitcher_id = row['pitcher']

    # Get the batter rating and instances (ABs)
    batter = {}
    if batter_id in batters:
        batter = batters[batter_id]
    else:
        batter = {
            'rating': 1500,
            'instances': 0
        }

    # Get the pitcher rating and instances (BFs)
    pitcher = {}
    if pitcher_id in pitchers:
        pitcher = pitchers[pitcher_id]
    else:
        pitcher = {
            'rating': 1500,
            'instances': 0
        }

    b_rating = batter['rating']
    p_rating = pitcher['rating']

    elo_change = 1 / (1 + 10 ** ((p_rating - b_rating) / 400))
    expected = league_average + (elo_change - 0.5) * 2 * league_average

    actual = OUTCOMES.get(row['pa_result'])
    
    # Volitity. Also try 40 (fewer PAs) or 16 (more PAs)
    k = 20

    # Batter Ratings
    batter['rating'] = b_rating + (k * (actual - expected))
    batter['instances'] += 1

    batters[batter_id] = batter

    # Pitcher Ratings
    pitcher['rating'] = p_rating + (k * ((1 - actual) - (1 - expected)))
    pitcher['instances'] += 1

    pitchers[pitcher_id] = pitcher

b_ratings_df = pd.DataFrame.from_dict(batters, orient='index')
b_ratings_df.index.name = 'batter_id'
b_ratings_df.reset_index()
print(b_ratings_df.loc[b_ratings_df['instances'].ge(100)].sort_values('rating', ascending=False).head())

p_ratings_df = pd.DataFrame.from_dict(pitchers, orient='index')
p_ratings_df.index.name = 'pitcher_id'
p_ratings_df.reset_index()
print(p_ratings_df.loc[p_ratings_df['instances'].ge(100)].sort_values('rating', ascending=False).head())


                rating  instances
batter_id                        
judga001   1462.596856        698
sprig001   1436.143151        600
horws001   1435.422130        421
liled001   1435.061071        358
jonej007   1433.121486        152
                 rating  instances
pitcher_id                        
millm004    1760.446990        250
chapa001    1757.300202        241
yamay001    1741.943720        701
sheee001    1733.816753        298
diaze006    1733.594216        293
